In [ ]:
import os
import scanpy as sc
import anndata as ad
import numpy as np
import subprocess
from sklearn.model_selection import train_test_split

# define your dataset name
dataset_name = ''

## RNA preprocess

In [ ]:
adata = sc.read_h5ad(f'/path/rna_data/rna.h5ad')
gene_list = np.loadtxt("/path/rna_data/rna_cellplm_gene_names.txt", dtype=str)
cellplm_gene_list = np.loadtxt("/path/rna_data/rna_cellplm_genes.txt", dtype=str)
rna_zero = ad.AnnData(np.zeros((1, len(gene_list)), dtype=float))  # to align atlas rna gene list
rna_zero.var_names = gene_list
rna_m = ad.concat([rna_zero, adata], axis=0, join='outer', fill_value=0)
rna_m = rna_m[1:, gene_list].copy()
rna_m.var_names = cellplm_gene_list
rna_m.var['gene_name'] = gene_list
rna_m.obs_names_make_unique()
rna_m.write_h5ad(f"/path/rna_data/rna_{dataset_name}.h5ad", compression="gzip")

## ATAC preprocess

### fragment file convert to mtx through cpeak

In [ ]:
cpeak_path = '/path'  # directory of cpeaks.py
main_script = 'cpeaks.py'
fragment_file = '/path/atac_data/fragments.bed.gz'
output_dir = '/path/atac_data'
reference_path = '/path'  # reference file directory of cpeaks_hg38.bed.gz
barcodes = '/path/rna_data/barcodes.txt'
subprocess.run(['python', main_script, "-f", fragment_file, "-b", barcodes, "--output_name", fragment_file.split(".bed")[0], "-o", output_dir, "-cpeaks", reference_path], cwd=cpeak_path)

### Align the peaks of the mtx file to the reference peaks of epiGen

In [ ]:
peaks = np.loadtxt(os.path.join(output_dir, f"cpeaks_hg38.bed"), delimiter="\t", dtype=str)  # unzip from cpeaks_hg38.bed.gz
peaks_var = np.array(["-".join(str(i) for i in peak) for peak in peaks])
mt = sc.read_mtx(f"{fragment_file.split('.bed')[0]}.mtx").T
mt.var_names = peaks_var
peaks = np.loadtxt(os.path.join(output_dir, f"peaks.bed"), delimiter="\t", dtype=str)
peaks_var = ["-".join(str(i) for i in peak) for peak in peaks]
atac_m = mt[:, peaks_var].copy()
atac_m.write_h5ad(os.path.join(output_dir, f"atac_{dataset_name}.h5ad"), compression="gzip")

## train/test split

In [ ]:
train_idx, test_idx = train_test_split(range(len(rna_m)), test_size=0.1, random_state=0)
np.save(f"/path/train_idx_{dataset_name}.npy", train_idx)
np.save(f"/path/test_idx_{dataset_name}.npy", test_idx)